# Feature Standardization: From Intuition to a Leakage-Safe Workflow

This notebook is an original, hands-on guide to feature standardization. It builds the idea from first principles, compares common scaling choices, and finishes with a complete scikit-learn workflow. The examples use reproducible data so you can inspect what each transformation does.

## Learning goals
- Explain why different feature units and ranges can affect some machine-learning algorithms.
- Compute and interpret z-scores.
- Use `StandardScaler` correctly, including on train/test data without leakage.
- Compare standardization with min-max and robust scaling.
- Decide when scaling is useful and how to reverse it.

> Standardization changes feature units; it does not make data normally distributed, remove outliers, or guarantee a better model.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
pd.set_option("display.precision", 3)
print("Imports ready; reproducible random state:", RANDOM_STATE)

Imports ready; reproducible random state: 42


## 1. Why put features on comparable scales?

Imagine a model using **annual income** (roughly tens of thousands) and **years of experience** (roughly single digits). A distance calculation, gradient-based optimizer, or regularization penalty may respond much more strongly to the numerically large income feature. The model can then behave as if units were importance.

Scaling changes numeric representation, not the underlying observations. It is especially relevant to distance-based methods (such as k-nearest neighbors and k-means), support-vector machines, PCA, and many regularized or gradient-based models. Decision trees and random forests generally do not require feature scaling because their splits depend on feature ordering, not feature magnitude.

In [ ]:
customers = pd.DataFrame({
    "annual_income": [32_000, 45_000, 58_000, 72_000, 95_000],
    "experience_years": [1, 3, 5, 8, 12],
})
print(customers)
print("\nOriginal feature standard deviations:\n", customers.std().round(2))

## 2. The z-score transformation

For a feature value $x$, standardization computes:

$$z = \frac{x - \mu}{\sigma}$$

Here, $\mu$ is the feature mean and $\sigma$ is its standard deviation, estimated from the data used to **fit** the scaler. Subtracting the mean centers values around zero; dividing by the standard deviation expresses differences in standard-deviation units. For the fitted data, each transformed feature has mean approximately 0 and standard deviation approximately 1.

The formula is applied independently to every column. A z-score of 2 means the observation is two standard deviations above that feature's fitted mean; it does not mean that the observation is two times larger.

In [ ]:
# Work through the arithmetic on one feature before using a library.
experience = customers["experience_years"].to_numpy(dtype=float)
mean_experience = experience.mean()
std_experience = experience.std(ddof=0)  # population convention used by StandardScaler
z_manual = (experience - mean_experience) / std_experience

print(f"mean = {mean_experience:.2f}; population std = {std_experience:.2f}")
print("manual z-scores:", np.round(z_manual, 3))
print(f"transformed mean = {z_manual.mean():.3f}; transformed std = {z_manual.std(ddof=0):.3f}")

## 3. Standardize several columns with `StandardScaler`

`fit` learns each column's mean and scale. `transform` applies those learned values. The scaler accepts a two-dimensional feature matrix; keep the column names alongside the transformed values when working with a DataFrame.

The mean and standard deviation are properties of the **fitting data**, not a guarantee about every future batch. New observations may transform to values outside any range seen during fitting, which is expected.

In [ ]:
scaler = StandardScaler()
customers_scaled = pd.DataFrame(
    scaler.fit_transform(customers),
    columns=customers.columns,
    index=customers.index,
)
print("Scaled values:\n", customers_scaled.round(3))
print("\nMeans learned by scaler:", dict(zip(customers.columns, scaler.mean_.round(2))))
print("Scales learned by scaler:", dict(zip(customers.columns, scaler.scale_.round(2))))
print("\nScaled-data means:\n", customers_scaled.mean().round(10))
print("Scaled-data population stds:\n", customers_scaled.std(ddof=0).round(10))

## 4. The most important rule: prevent data leakage

Do **not** fit a scaler on the complete dataset before splitting into train and test sets. That would let test-set statistics influence the training process. Instead:

1. Split the raw features and labels.
2. Fit preprocessing only on the training features.
3. Use that already-fitted transformer on both training and test features.

A scikit-learn `Pipeline` makes this order explicit and helps ensure cross-validation also refits preprocessing inside each training fold. Never call `fit_transform` on test data.

In [ ]:
# Create a binary classification example with deliberately different feature scales.
X_base, y = make_classification(
    n_samples=600,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    random_state=RANDOM_STATE,
)
X = pd.DataFrame(X_base, columns=["signal_a", "signal_b", "signal_c", "signal_d"])
X["signal_a"] *= 1_000  # change units, not its predictive information
X["signal_b"] *= 0.01

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

model = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)  # scaler is fitted on X_train only
predictions = model.predict(X_test)  # pipeline transforms X_test with training statistics
print(f"Train rows: {len(X_train)}; test rows: {len(X_test)}")
print(f"Test accuracy: {accuracy_score(y_test, predictions):.3f}")
print("Scaler mean (training features only):", model.named_steps["scale"].mean_.round(3))

## 5. Compare scaling methods

- **StandardScaler:** subtracts the mean and divides by standard deviation. Useful as a common default for algorithms sensitive to scale, but the mean and standard deviation are sensitive to outliers.
- **MinMaxScaler:** maps the fitted minimum and maximum to a chosen interval (by default 0 to 1). It preserves the shape of the distribution but extreme values can compress most observations into a small range. Future values may fall outside the fitted interval.
- **RobustScaler:** centers by the median and scales by the interquartile range (IQR). It is less affected by extreme observations, but does not eliminate their influence on every downstream algorithm.

There is no universally best scaler. Choose based on the data and model, and fit the choice inside the training pipeline.

In [ ]:
values = np.array([10, 11, 12, 13, 14, 15, 16, 100], dtype=float).reshape(-1, 1)
methods = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}
comparison = pd.DataFrame({
    name: transformer.fit_transform(values).ravel()
    for name, transformer in methods.items()
})
comparison.insert(0, "original", values.ravel())
print(comparison.round(3))
print("\nCompare the first seven rows: the value 100 changes the standard and min-max scales much more than the robust scale.")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
for ax, column in zip(axes, comparison.columns):
    ax.scatter(comparison[column], np.arange(len(comparison)), alpha=0.8)
    ax.set_title(column)
    ax.set_xlabel("Transformed value")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Observation index")
fig.suptitle("One extreme value affects each scaler differently")
fig.tight_layout()
plt.show()

## 6. Reverse the transformation

If you need to report predictions or transformed measurements in their original units, use the **same fitted scaler** and `inverse_transform`. This does not recover information lost by other operations such as rounding or clipping; it simply reverses this affine scaling.

In [ ]:
scaled_values = scaler.transform(X_train.iloc[:3])
restored_values = scaler.inverse_transform(scaled_values)
restored = pd.DataFrame(restored_values, columns=X.columns, index=X_train.index[:3])
print("Original rows:\n", X_train.iloc[:3].round(3))
print("\nAfter transform then inverse_transform:\n", restored.round(3))
print("\nMaximum absolute reconstruction error:", np.abs(restored_values - X_train.iloc[:3].to_numpy()).max())

## 7. Practical checklist and common mistakes

- **Scale numeric inputs, not labels.** For classification, do not standardize the target class values.
- **Fit only on training data.** A pipeline is the safest default when estimating generalization or using cross-validation.
- **Apply the same fitted transformer at inference time.** Persist the complete fitted pipeline with the model.
- **Inspect missing values and feature types first.** `StandardScaler` expects numeric values; imputation and categorical encoding should be handled in the preprocessing pipeline where needed.
- **Remember outliers.** Standard scaling does not cap, remove, or robustly neutralize them.
- **Do not scale automatically.** Tree-based estimators usually do not need it; evaluate the preprocessing choices that match your algorithm.
- **Keep sparse data sparse when appropriate.** Centering a sparse matrix makes it dense; configure the scaler deliberately for sparse workflows.


### Key takeaway
Standardization makes feature scales comparable by learning a mean and standard deviation per feature. The transformation is simple; the critical practice is fitting it only on training data and reusing that fitted transformation consistently.